# EpiAI 一站式教程
从原始数据到部署预测 + 可视化

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
from pathlib import Path

from EpiAI import ForecastTask
from EpiAI.dataset import TimeSplit
from EpiAI.viz import plot_test_predictions, plot_forecast

## 1. 加载数据

In [ ]:
DATA_PATH = Path.home() / "infective_disease" / "pingtai" / "merged_clean.csv"
DISEASE = "登革热 Dengue fever"

df_raw = pd.read_csv(DATA_PATH)
df_raw["Year/Month"] = pd.to_datetime(df_raw["Year/Month"])
df = (
    df_raw.query("Diseases == @DISEASE")
    [["Year/Month", "Case number"]]
    .rename(columns={"Year/Month": "time", "Case number": "cases"})
    .reset_index(drop=True)
)
df = df[df["time"].dt.year >= 2010].reset_index(drop=True)
print(f"数据: {len(df)} 行, {df['time'].min().date()} → {df['time'].max().date()}")

## 2. 定义预测任务

In [ ]:
task = ForecastTask(
    data=df,
    target="cases",
    time_col="time",
    freq="MS",
    horizon=12,
    split=TimeSplit(train_end="2024-05-01", val_end="2025-05-01"),
)
print(f"Task: {task}")
print(f"季节周期: {task.freq_to_season_length()} 期/年")

## 3. 批量训练 + 排行榜

In [ ]:
vault = task.compare([
    "sf_auto_ets",
    "sf_holt_winters",
    "sf_theta",
    "prophet",
    "serfling",
])
print("\n排行榜（按 R² 排序）:")
display(vault.summary())

## 4. 测试集预测对比图

真实值（黑线） vs 各模型在测试集上的预测值（散点+虚线），图例附带 Pearson r。

In [ ]:
plot_test_predictions(
    bundle=task.build_bundle(),
    vault=vault,
    title=f"{DISEASE} 测试集预测对比",
)

## 5. 部署 + 风险评分

In [ ]:
best_model = vault.best("R2")
print(f"最佳模型: {best_model}")

runtime = task.deploy(best_model, vault, risk=["quantile", "zscore"])
print(f"Runtime: {runtime}")
print(f"风险算法: {list(runtime.risk_scorers.keys())}")

## 6. 未来预测图

历史曲线（最近 20 期） + 多模型未来预测 + 风险等级。

In [ ]:
hist = runtime.data_table.tail(20)
pred = runtime.predict(horizon=12, level=95)

plot_forecast(
    hist_df=hist,
    pred_df=pred,
    metrics_df=vault.metrics,
    title=f"{DISEASE} 未来 12 期预测",
)